In [ ]:
# Install libraries
!pip install pandas scikit-learn beautifulsoup4 transformers

In [ ]:
import pandas as pd
import re
from bs4 import BeautifulSoup
from sklearn.model_selection import train_test_split
from transformers import DistilBertTokenizerFast

**Load Dataset**

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import pandas as pd

df = pd.read_csv("IMDB Dataset.csv")

df.head()

In [ ]:
df.columns

**Clean text**

In [ ]:
from bs4 import BeautifulSoup
import re

def clean_text(text):

    # handle missing values
    text = str(text)

    # remove HTML tags
    text = BeautifulSoup(text, "html.parser").get_text()

    # remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

# apply cleaning
df["clean_review"] = df["review"].apply(clean_text)

**Convert labels**

In [ ]:
df["label"] = df["sentiment"].map({
    "negative": 0,
    "positive": 1
})

**Train / validation / test split**

In [ ]:
from sklearn.model_selection import train_test_split

train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df["clean_review"].tolist(),
    df["label"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts,
    temp_labels,
    test_size=0.5,
    random_state=42,
    stratify=temp_labels
)

**Tokenization for DistilBERT**

In [ ]:
from transformers import DistilBertTokenizerFast

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=256
)

val_encodings = tokenizer(
    val_texts,
    truncation=True,
    padding=True,
    max_length=256
)

test_encodings = tokenizer(
    test_texts,
    truncation=True,
    padding=True,
    max_length=256
)

In [ ]:
print(train_encodings.keys())
print(len(train_encodings["input_ids"]))
print(len(train_encodings["attention_mask"]))
print(train_encodings["input_ids"][0][:20])
print(train_encodings["attention_mask"][0][:20])

In [ ]:
import torch

class IMDbDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
train_dataset = IMDbDataset(train_encodings, train_labels)
val_dataset = IMDbDataset(val_encodings, val_labels)
test_dataset = IMDbDataset(test_encodings, test_labels)

In [ ]:
print(train_dataset[0])

In [ ]:
import torch

torch.save(train_dataset, "train_dataset.pt")
torch.save(val_dataset, "val_dataset.pt")
torch.save(test_dataset, "test_dataset.pt")

In [ ]:
from google.colab import files

files.download("train_dataset.pt")
files.download("val_dataset.pt")
files.download("test_dataset.pt")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch

torch.save(train_dataset, "/content/drive/MyDrive/train_dataset.pt")
torch.save(val_dataset, "/content/drive/MyDrive/val_dataset.pt")
torch.save(test_dataset, "/content/drive/MyDrive/test_dataset.pt")

In [ ]:
# ==============================
# Logistic Regression Baseline
# ==============================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# TF-IDF Vectorization
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2)
)

X_train = tfidf.fit_transform(train_texts)
X_val = tfidf.transform(val_texts)
X_test = tfidf.transform(test_texts)

# Logistic Regression Model
log_reg = LogisticRegression(max_iter=1000)

# Train model
log_reg.fit(X_train, train_labels)

# Predictions
y_pred = log_reg.predict(X_test)

# Evaluation Metrics
print("Accuracy:", accuracy_score(test_labels, y_pred))
print("Precision:", precision_score(test_labels, y_pred))
print("Recall:", recall_score(test_labels, y_pred))
print("F1-score:", f1_score(test_labels, y_pred))

# Detailed Classification Report
print("\nClassification Report:\n")
print(classification_report(
    test_labels,
    y_pred,
    target_names=["negative", "positive"]
))

# Confusion Matrix
print("\nConfusion Matrix:\n")
print(confusion_matrix(test_labels, y_pred))

In [ ]:
# ==============================
# DistilBERT Model Training
# ==============================

!pip install transformers evaluate -q

In [ ]:
import torch
import numpy as np
from transformers import DistilBertForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

In [ ]:
# Create PyTorch Dataset Class

class IMDbDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
# Create Dataset Objects

train_dataset = IMDbDataset(train_encodings, train_labels)
val_dataset = IMDbDataset(val_encodings, val_labels)
test_dataset = IMDbDataset(test_encodings, test_labels)

In [ ]:
# Load DistilBERT Model

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

In [ ]:
# Evaluation Metrics

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [ ]:
# Training Settings

training_args = TrainingArguments(
    output_dir="./distilbert_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100,
    load_best_model_at_end=True
)

In [ ]:
# Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
# Train DistilBERT

trainer.train()

In [ ]:
test_results = trainer.evaluate(test_dataset)
print(test_results)

**get predictions**

In [ ]:
predictions = trainer.predict(test_dataset)

y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

In [ ]:
print(classification_report(
    y_true,
    y_pred,
    target_names=["negative", "positive"]
))

In [ ]:
print(confusion_matrix(y_true, y_pred))

In [ ]:
trainer.save_model("./distilbert_imdb_model")
tokenizer.save_pretrained("./distilbert_imdb_model")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
import numpy as np

cm = np.array([
    [2284, 216],
    [184, 2316]
])

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Negative", "Positive"]
)

disp.plot(cmap="Blues")
plt.title("DistilBERT Confusion Matrix")
plt.show()

In [ ]:
print(model)

In [ ]:
import matplotlib.pyplot as plt

models = ["Logistic Regression", "DistilBERT", "RoBERTa"]
accuracy = [89.28, 92.0, 91.0]

plt.figure(figsize=(8,5))

plt.bar(models, accuracy)

plt.ylabel("Accuracy (%)")
plt.title("Model Performance Comparison")

plt.ylim(85, 95)

for i, v in enumerate(accuracy):
    plt.text(i, v + 0.2, str(v) + "%", ha='center')

plt.show()

In [ ]:
!pip install transformers torch matplotlib seaborn -q

In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

In [ ]:
review = "This was one of the worst movies I have ever watched."

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
model_path = "/content/drive/MyDrive/distilbert_imdb_model"

tokenizer = DistilBertTokenizerFast.from_pretrained(model_path)

model = DistilBertForSequenceClassification.from_pretrained(
    model_path,
    output_attentions=True,
    attn_implementation="eager"
)

model.eval()

In [ ]:
inputs = tokenizer(
    review,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=128
)

In [ ]:
with torch.no_grad():
    outputs = model(**inputs)

predicted_class = torch.argmax(outputs.logits, dim=1).item()
label = "Positive" if predicted_class == 1 else "Negative"

print("Predicted sentiment:", label)

In [ ]:
attentions = outputs.attentions

last_layer_attention = attentions[-1]
avg_attention = last_layer_attention.mean(dim=1).squeeze(0)

cls_attention = avg_attention[0].detach().cpu().numpy()
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

In [ ]:
plt.figure(figsize=(12, 4))

sns.heatmap(
    [cls_attention],
    xticklabels=tokens,
    yticklabels=["Attention"],
    cmap="Blues"
)

plt.xticks(rotation=45, ha="right")
plt.title("DistilBERT Attention Map")
plt.show()